# 03 — Feature Engineering

**Purpose:** Transform raw data into a model-ready feature matrix.

The pipeline is config-driven: adding a new feature requires only:
1. Add its name to `config/features.yaml` under the right group
2. Implement its formula in `src/feature_engineering.py → _compute_derived_features()`
3. Re-run this notebook

Steps:
1. Load quality-approved data
2. Compute derived features
3. Impute missing values
4. Encode categoricals
5. Inspect final feature matrix
6. Save train/test splits

---
**Inputs:** `data/raw/survival_data.parquet`, `data/raw/quality_flag.json`  
**Outputs:** `data/processed/X_train.parquet`, `data/processed/X_test.parquet`,  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`data/processed/y_train.parquet`, `data/processed/y_test.parquet`,  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`models/encoders.pkl`

In [ ]:
import sys, json
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data_utils import load_data, save_data, load_config, DATA_RAW, MODELS_DIR
from src.feature_engineering import build_feature_matrix, summarize_features, get_feature_groups

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 40)
print('Ready.')

## 1. Gate Check — Data Quality Approval

In [ ]:
flag_path = DATA_RAW / 'quality_flag.json'
if flag_path.exists():
    with open(flag_path) as f:
        flag = json.load(f)
    print(f"Quality verdict : {flag['verdict']}")
    if not flag['approved']:
        raise RuntimeError('Dataset not approved. Fix data quality issues in notebook 02 first.')
else:
    print('Warning: quality_flag.json not found. Run notebook 02 first for a full quality check.')

In [ ]:
cfg = load_config()
DURATION_COL = cfg['target']['duration_col']
EVENT_COL    = cfg['target']['event_col']
TEST_SIZE    = cfg['experiment']['test_size']
SEED         = cfg['experiment']['random_state']

df = load_data('survival_data', stage='raw')
print(f'Shape: {df.shape}')

## 2. Build Feature Matrix

In [ ]:
# Separate targets before engineering (avoids leakage)
duration = df[DURATION_COL].copy()
event    = df[EVENT_COL].copy()
df_raw   = df.drop(columns=[DURATION_COL, EVENT_COL])

X, encoders = build_feature_matrix(df_raw, cfg, fit_encoders=True)

summarize_features(X, cfg)

## 3. Feature Group Inspection

In [ ]:
groups = get_feature_groups(cfg, X)
for group_name, feats in groups.items():
    print(f'\n── {group_name.upper()} FEATURES ({len(feats)}) ──')
    if feats:
        display(X[feats].describe().T.round(3))

## 4. Verify No Remaining NaN

In [ ]:
nan_counts = X.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]

if len(nan_cols) == 0:
    print('✓ Feature matrix has zero NaN values.')
else:
    print('✗ NaN values remain:')
    print(nan_cols)

## 5. Feature Correlation Matrix

In [ ]:
corr = X.corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.3,
            annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix (engineered)', fontsize=12)
plt.tight_layout()
plt.show()

# Flag highly correlated pairs
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i,j]) > 0.85:
            high_corr.append((corr.columns[i], corr.columns[j], round(corr.iloc[i,j], 3)))

if high_corr:
    print('\n⚠️  Highly correlated pairs (|r| > 0.85) — consider removing one:')
    for a, b, r in high_corr:
        print(f'   {a} × {b}: r = {r}')
else:
    print('\n✓ No highly correlated pairs found.')

## 6. Train / Test Split

In [ ]:
X_train, X_test, dur_train, dur_test, ev_train, ev_test = train_test_split(
    X, duration, event,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=event if cfg['experiment']['stratify_by_event'] else None,
)

print(f'Train : {X_train.shape}  |  event rate = {ev_train.mean():.2%}')
print(f'Test  : {X_test.shape}  |  event rate = {ev_test.mean():.2%}')

## 7. Save Outputs

In [ ]:
# Feature matrices
save_data(X_train.reset_index(drop=True), 'X_train', stage='processed')
save_data(X_test.reset_index(drop=True),  'X_test',  stage='processed')

# Targets
y_train = pd.DataFrame({'duration': dur_train.values, 'event': ev_train.values})
y_test  = pd.DataFrame({'duration': dur_test.values,  'event': ev_test.values})
save_data(y_train, 'y_train', stage='processed')
save_data(y_test,  'y_test',  stage='processed')

# Fitted encoders (reuse for inference)
enc_path = MODELS_DIR / 'encoders.pkl'
joblib.dump(encoders, enc_path)
print(f'Encoders saved → {enc_path}')

print('\n✓ Done. Proceed to 04_feature_selection.ipynb')